# Question 1 — Interactive waste-bin map

**Aditi Saha · HS25M101**

Folium map of waste-bin observations collected with QField at IIT Tirupati.
Each point shows a tooltip, a popup with the collector name, and a photograph
when one is available.

Photographs are stored in `web/photos` using the QField file names, so they
remain visible after the map is hosted.


In [ ]:
# 1. Import libraries
import geopandas as gpd
import folium
from branca.element import Element


In [ ]:
# 2. Load the waste-bin GeoPackage
data_path = r"C:\Users\Ā\Documents\python_class\HS25M101\HS25M101.gpkg"

dustbins = gpd.read_file(
    data_path,
    layer="notes_point"
)

attachments = gpd.read_file(
    data_path,
    layer="notes_attachments"
)


In [ ]:
# 3. Check CRS and convert to EPSG:4326
print(dustbins.crs)

dustbins = dustbins.to_crs(epsg=4326)

dustbins["timestamp"] = dustbins["timestamp"].astype(str)
dustbins["title"] = dustbins["title"].fillna("No title")
dustbins["note"] = dustbins["note"].fillna("No note")
dustbins["collector"] = "Aditi Saha"


In [ ]:
# 4. Join photograph names to each point
photos = attachments[["note_uuid", "media"]].drop_duplicates("note_uuid")

dustbins = dustbins.merge(
    photos,
    left_on="uuid",
    right_on="note_uuid",
    how="left"
)

dustbins["photo_name"] = dustbins["media"].astype(str).str.split("/").str[-1]
dustbins.loc[dustbins["media"].isna(), "photo_name"] = ""

dustbins[["title", "note", "collector", "photo_name"]].head()


In [ ]:
# 5. Build popup text for each point
popups = []

for i, row in dustbins.iterrows():
    html = (
        "<div style='width:240px;font-size:13px;line-height:1.4'>"
        + "<b>" + str(row["title"]) + "</b><br>"
        + "<b>Collected by:</b> " + str(row["collector"]) + "<br>"
        + "<b>Note:</b> " + str(row["note"]) + "<br>"
        + "<b>Recorded at:</b> " + str(row["timestamp"])
    )
    if row["photo_name"] != "":
        html = (
            html
            + "<br><img src='photos/" + row["photo_name"]
            + "' style='margin-top:8px;width:100%;border-radius:4px'>"
        )
    else:
        html = html + "<br><i>No photograph attached</i>"
    html = html + "</div>"
    popups.append(html)

dustbins["popup"] = popups


In [ ]:
# 6. Calculate map centre
center = [
    dustbins.geometry.y.mean(),
    dustbins.geometry.x.mean()
]
print(center)


In [ ]:
# 7. Create the Folium map
m = folium.Map(
    location=center,
    zoom_start=16,
    tiles=None,
    control_scale=True
)

folium.TileLayer(
    tiles="OpenStreetMap",
    name="Streets"
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite"
).add_to(m)


In [ ]:
# 8. Add waste-bin points with tooltip and popup
bins = folium.FeatureGroup(name="Waste bins")

for i, row in dustbins.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=7,
        fill=True,
        fill_opacity=0.9,
        weight=1,
        color="white",
        fill_color="#377eb8",
        tooltip=str(row["title"]),
        popup=folium.Popup(row["popup"], max_width=280)
    ).add_to(bins)

bins.add_to(m)


In [ ]:
# 9. Add title, legend, north arrow, layer control, and fit the map

def add_north_arrow(m, lat, lon):
    """Add a simple north arrow to a Folium map."""
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=(
                "<div style='font-size:16px; font-weight:bold; "
                "text-align:center; color:black'>"
                "↑<br>N</div>"
            )
        )
    ).add_to(m)


title_html = """
<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);
z-index:9999;background:white;padding:8px 14px;border-radius:6px;
box-shadow:0 1px 4px rgba(0,0,0,0.3);text-align:center;
font-family:Arial;font-size:14px">
<b>Waste-bin observations, IIT Tirupati</b><br>
<span style="font-size:12px">Collected by Aditi Saha · QField · Sep 2026</span>
</div>
"""

legend_html = """
<div style="position:fixed;bottom:40px;left:12px;z-index:9999;
background:white;padding:8px 12px;border-radius:6px;
box-shadow:0 1px 4px rgba(0,0,0,0.3);font-family:Arial;font-size:12px">
<b>Legend</b><br>
<span style="display:inline-block;width:10px;height:10px;
border-radius:50%;background:#377eb8;margin-right:6px"></span>
Waste-bin observation
</div>
"""

m.get_root().html.add_child(Element(title_html))
m.get_root().html.add_child(Element(legend_html))

north_lat = dustbins.geometry.y.max()
north_lon = dustbins.geometry.x.mean()
add_north_arrow(m, north_lat, north_lon)

folium.LayerControl().add_to(m)

bounds = [
    [dustbins.geometry.y.min(), dustbins.geometry.x.min()],
    [dustbins.geometry.y.max(), dustbins.geometry.x.max()]
]
m.fit_bounds(bounds)

m


In [ ]:
# 10. Save the map as HTML for the website
m.save("web/waste-bins.html")
print("Saved web/waste-bins.html")
